<a href="https://colab.research.google.com/github/Krisnawati12/Analisis-Data/blob/main/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np

# ==========================================================
# 1. DATA UNDERSTANDING
# ==========================================================
# a. Load dataset
df = pd.read_csv('all_months_clean.csv', sep=';') # Added sep=';' to handle potential semicolon delimiter

# b. Tampilkan informasi dasar
print("--- 5 BARIS PERTAMA ---")
display(df.head())

print("\n--- JUMLAH BARIS DAN KOLOM ---")
print(f"Dataset memiliki {df.shape[0]} baris dan {df.shape[1]} kolom.")

print("\n--- NAMA KOLOM ---")
print(df.columns.tolist())


# c. Identifikasi Variabel (Berdasarkan analisis tipe data)
print("\n--- IDENTIFIKASI VARIABEL ---")
# Target: is_cancelled (akan dibuat di tahap preprocessing)
# Fitur Numerik: purchase_price, shipping_fee, quantity
# Fitur Kategorikal: payment_method, product_category, order_status
print("Target: is_cancelled (dari Status Pesanan)") # Updated to reflect actual column name
print("Numerik: Ongkos Kirim Dibayar oleh Pembeli, total_qty") # Updated to reflect actual column names. 'purchase_price' is not directly available.
print("Kategorikal: Metode Pembayaran, product_categories, Provinsi") # Updated to reflect actual column names


# ==========================================================
# 2. DATA PREPROCESSING
# ==========================================================
# a. Buat fitur is_cancelled dan is_cod
df['is_cancelled'] = df['Status Pesanan'].apply(lambda x: 1 if x == 'Batal' else 0) # Corrected column name and cancellation value
df['is_cod'] = df['Metode Pembayaran'].apply(lambda x: 1 if x == 'COD (Bayar di Tempat)' else 0) # Corrected column name and COD value

# b. Ekstrak jam dari kolom waktu (asumsi nama kolom: order_time)
df['order_hour'] = pd.to_datetime(df['Waktu Pesanan Dibuat']).dt.hour # Corrected column name

# c. Fitur tambahan: Shipping Ratio (Perbandingan ongkir dengan harga barang)
# df['shipping_ratio'] = df['Ongkos Kirim Dibayar oleh Pembeli'] / df['purchase_price'] # Commented out: 'purchase_price' is undefined.

# d. Simpan dataset modifikasi (GANTI NPM SESUAI PUNYAMU)
df.to_csv('data_processed_24782001.csv', index=False)
print("\n[INFO] Dataset terbaru telah disimpan.")

# 2c. Fitur tambahan: Pengelompokan Ongkir (Shipping Group)
# Kita bagi ongkir jadi 3 kelompok: Murah, Sedang, Mahal
df['shipping_group'] = pd.qcut(df['Perkiraan Ongkos Kirim'], q=3, labels=['Murah', 'Sedang', 'Mahal'])

# Baru panggil lagi display-nya
display(df[['is_cancelled', 'is_cod', 'order_hour', 'shipping_group']].head())
# ==========================================================
# 4. ANALISIS POLA (Menjawab No 3 & 4)
# ==========================================================
print("\n--- 4a. ANALISIS COD VS NON-COD ---")
pola_cod = df.groupby('is_cod')['is_cancelled'].mean() * 100
print(pola_cod)

# ==========================================================
# 4b. HUBUNGAN ONGKOS KIRIM DENGAN PEMBATALAN (SOLUSI FIX)
# ==========================================================
print("\n--- 4b. HUBUNGAN ONGKOS KIRIM DENGAN PEMBATALAN ---")

# Gunakan 'Perkiraan Ongkos Kirim' karena datanya lebih bervariasi (tidak banyak angka 0)
data_ongkir = df['Perkiraan Ongkos Kirim'].dropna()

# Kita bagi jadi 3 kategori berdasarkan kuartil (Murah, Sedang, Mahal)
# qcut akan membagi jumlah data sama rata ke 3 kelompok ini
df['shipping_group'] = pd.qcut(data_ongkir, 3, labels=['Murah', 'Sedang', 'Mahal'])

pola_ongkir = df.groupby('shipping_group', observed=True)['is_cancelled'].mean() * 100
print(pola_ongkir)

# Tambahan: Tampilkan rentang harga tiap kelompok supaya jelas di laporan
print("\nRentang harga tiap kategori:")
print(df.groupby('shipping_group', observed=True)['Perkiraan Ongkos Kirim'].agg(['min', 'max']))

print("\n--- 4c. HUBUNGAN JUMLAH PRODUK DENGAN PEMBATALAN ---")
pola_qty = df.groupby('total_qty')['is_cancelled'].mean() * 100 # Corrected column name
print(pola_qty)


# ==========================================================
# 5. SEGMENTASI DATA
# ==========================================================
print("\n--- 5. SEGMENTASI RISIKO TERTINGGI ---")
# Segmentasi berdasarkan Metode Pembayaran dan Kategori Ongkir
segmentasi = df.groupby(['is_cod', 'shipping_group'], observed=True)['is_cancelled'].mean() * 100
print(segmentasi)


# ==========================================================
# 6. ANALISIS KONFLIK
# ==========================================================
print("\n--- 6. ANALISIS KONFLIK DATA ---")
# Data sesuai pola: COD yang dibatalkan
pola_umum = df[(df['is_cod'] == 1) & (df['is_cancelled'] == 1)].head(1)
# Data bertentangan: COD dengan ongkir mahal tapi TETAP BERHASIL (tidak batal)
pola_konflik = df[(df['is_cod'] == 1) & (df['is_cancelled'] == 0) & (df['Ongkos Kirim Dibayar oleh Pembeli'] > df['Ongkos Kirim Dibayar oleh Pembeli'].median())].head(1) # Corrected column name

print("\nData Sesuai Pola Umum:")
display(pola_umum)
print("\nData Bertentangan (Konflik):")
display(pola_konflik)

--- 5 BARIS PERTAMA ---


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024.xlsx
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024.xlsx
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024.xlsx
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024.xlsx
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024.xlsx



--- JUMLAH BARIS DAN KOLOM ---
Dataset memiliki 20848 baris dan 19 kolom.

--- NAMA KOLOM ---
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', 'source_file']

--- IDENTIFIKASI VARIABEL ---
Target: is_cancelled (dari Status Pesanan)
Numerik: Ongkos Kirim Dibayar oleh Pembeli, total_qty
Kategorikal: Metode Pembayaran, product_categories, Provinsi

[INFO] Dataset terbaru telah disimpan.


,is_cancelled,is_cod,order_hour,shipping_group
0,0,0,0.0,Sedang
1,0,1,1.0,Sedang
2,0,0,4.0,Murah
3,0,1,4.0,Mahal
4,1,1,6.0,Murah



--- 4a. ANALISIS COD VS NON-COD ---
is_cod
0    13.813104
1    13.381869
Name: is_cancelled, dtype: float64

--- 4b. HUBUNGAN ONGKOS KIRIM DENGAN PEMBATALAN ---
shipping_group
Murah     10.238429
Sedang    12.042728
Mahal     18.550261
Name: is_cancelled, dtype: float64

Rentang harga tiap kategori:
                  min     max
shipping_group               
Murah               1    9000
Sedang           9100   16500
Mahal           16700  959200

--- 4c. HUBUNGAN JUMLAH PRODUK DENGAN PEMBATALAN ---
total_qty
1       14.221970
2       10.850504
3       12.778603
4       13.451777
5       14.065511
6       17.127072
7       20.000000
8       13.725490
9       25.000000
10      16.504854
11      27.272727
12      16.666667
13      11.764706
14       0.000000
15      18.918919
16       0.000000
17      28.571429
18      57.142857
19       0.000000
20      10.784314
21       0.000000
22      66.666667
23       0.000000
24      20.000000
25       0.000000
26       0.000000
27       0.00000

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,...,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file,is_cancelled,is_cod,order_hour,shipping_group
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,...,0,0,0,8000,2024-04-01 06:12,AprilSales2024.xlsx,1,1,6.0,Murah



Data Bertentangan (Konflik):


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,...,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file,is_cancelled,is_cod,order_hour,shipping_group
18,ORD_0000019,2,1200,0,0,"Celengan, Peralatan Kamar Mandi",2,Selesai,NaN,Agen SPX Express,...,3000,20000,35112,23000,2024-04-01 13:07,AprilSales2024.xlsx,0,1,13.0,Mahal
